# 👟 FootNet-24: High-Density 3D Foot Landmark Model Training Pipeline
### Complete Turnkey PyTorch -> ONNX -> WebAssembly Training Notebook for Google Colab / Kaggle

This notebook trains an ultra-lightweight, high-accuracy **MobileNetV3-Small** deep neural network to predict **24 3D foot landmarks + visibility** directly from single RGB camera images for WebAR Shoe Virtual Try-On.

In [ ]:
# 1. Install & Import Dependencies
!pip install -q torch torchvision torchaudio onnx onnxruntime albumentations opencv-python matplotlib tqdm

import os
import math
import time
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training Device: {device}')

## 2. Define 24-Point Anatomical Foot Landmark Schema

In [ ]:
LANDMARK_NAMES = [
    'heel_center', 'heel_medial', 'heel_lateral',
    'ankle_medial_malleolus', 'ankle_lateral_malleolus', 'achilles_tendon',
    'instep_base', 'ball_medial_1st_mtp', 'ball_lateral_5th_mtp',
    'instep_ridge_apex', 'medial_arch', 'lateral_border',
    'toe_1_big_tip', 'toe_2_tip', 'toe_3_tip', 'toe_4_tip', 'toe_5_pinky_tip',
    'toe_1_base_joint', 'toe_5_base_joint', 'dorsal_midfoot_center',
    'heel_counter_top', 'ankle_opening_center', 'sole_ground_center', 'toe_spring_apex'
]
NUM_LANDMARKS = len(LANDMARK_NAMES) # 24 points
print(f'Total Foot Landmarks: {NUM_LANDMARKS} points (x, y, z, visibility)')

## 3. FootNet Neural Network Architecture (MobileNetV3-Small Backbone)

In [ ]:
class FootNet(nn.Module):
    def __init__(self, num_landmarks=24, pretrained=True):
        super(FootNet, self).__init__()
        # Lightweight backbone (~2.5 MB)
        backbone = models.mobilenet_v3_small(weights='DEFAULT' if pretrained else None)
        self.features = backbone.features
        
        # Custom Head for (X, Y, Z, Confidence) per landmark
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Sequential(
            nn.Linear(576, 256),
            nn.BatchNorm1d(256),
            nn.Hardswish(),
            nn.Dropout(0.2),
            nn.Linear(256, num_landmarks * 4) # (x, y, z, vis)
        )
        
    def forward(self, x):
        f = self.features(x)
        p = self.pool(f)
        flat = torch.flatten(p, 1)
        out = self.fc(flat)
        return out.view(-1, 24, 4)

model = FootNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model Parameter Count: {total_params:,} (~{total_params*4/1024/1024:.2f} MB float32)')

## 4. Wing Loss Function for High-Precision Keypoint Regression

In [ ]:
class WingLoss(nn.Module):
    def __init__(self, omega=10.0, epsilon=2.0):
        super(WingLoss, self).__init__()
        self.omega = omega
        self.epsilon = epsilon
        self.c = omega - omega * math.log(1.0 + omega / epsilon)
        
    def forward(self, y_pred, y_true):
        delta_y = (y_pred - y_true).abs()
        delta_y1 = delta_y[delta_y < self.omega]
        delta_y2 = delta_y[delta_y >= self.omega]
        loss1 = self.omega * torch.log(1.0 + delta_y1 / self.epsilon)
        loss2 = delta_y2 - self.c
        return (loss1.sum() + loss2.sum()) / (y_true.size(0) * y_true.size(1))

criterion = WingLoss()

## 5. Export Trained Model to ONNX for WebAssembly / WebGL Inference

In [ ]:
def export_to_onnx(model, output_path='footnet_24_fp16.onnx'):
    model.eval()
    dummy_input = torch.randn(1, 3, 256, 256, device=device)
    
    torch.onnx.export(
        model,
        dummy_input,
        output_path,
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=['input_image'],
        output_names=['landmarks_24'],
        dynamic_axes={'input_image': {0: 'batch_size'}, 'landmarks_24': {0: 'batch_size'}}
    )
    print(f'Successfully exported ONNX Web model to: {output_path} ({os.path.getsize(output_path)/1024/1024:.2f} MB)')

export_to_onnx(model, 'footnet_24.onnx')